# 🛰️ GeoSight — Bengaluru Land Cover Change Analysis
> **Google Earth Engine + SAM 2 + Spectral Indices Pipeline**
>
> Author: Portfolio Project | CMR Institute of Technology
>
> Region: Bengaluru, India | Sensor: Sentinel-2 L2A | Period: 2019 → 2024


## 0. Setup

In [ ]:
import os, sys, ee, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from dotenv import load_dotenv

# Add src modules
sys.path.insert(0, '.')
from src.spectral import calculate_ndvi, calculate_ndwi, calculate_ndbi
from src.change_detect import identify_deforestation, identify_urbanization
from src.report import plot_spectral_index, generate_area_stats

load_dotenv()
ee.Initialize(project=os.environ.get('EE_PROJECT_ID', 'geosight-project'))
print('✅ Earth Engine connected')

## 1. Define Region of Interest — Bengaluru

In [ ]:
import leafmap

# Interactive map — zoom and draw your own ROI if you want!
m = leafmap.Map(center=[12.97, 77.59], zoom=10)
m.add_basemap('CartoDB.DarkMatter')
m

In [ ]:
ROI = ee.Geometry.Rectangle([77.45, 12.85, 77.75, 13.10])
print('ROI defined: Bengaluru metro area (30km x 25km)')

## 2. Load Sentinel-2 Composites (2019 & 2024)

In [ ]:
def build_composite(year):
    """Median cloud-free Sentinel-2 SR composite for a full year."""
    return (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(ROI)
        .filterDate(f'{year}-01-01', f'{year}-12-31')
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 15))
        .select(['B2', 'B3', 'B4', 'B8', 'B11'])
        .median()
    )

img_2019 = build_composite('2019')
img_2024 = build_composite('2024')

# Scene counts
n_2019 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(ROI).filterDate('2019-01-01', '2019-12-31').filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 15)).size().getInfo()
n_2024 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(ROI).filterDate('2024-01-01', '2024-12-31').filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 15)).size().getInfo()

print(f'✅ 2019: {n_2019} cloud-free scenes | 2024: {n_2024} cloud-free scenes')

## 3. Compute Spectral Indices (GEE-side)

In [ ]:
ndvi_2019 = img_2019.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndvi_2024 = img_2024.normalizedDifference(['B8', 'B4']).rename('NDVI')

ndwi_2019 = img_2019.normalizedDifference(['B3', 'B8']).rename('NDWI')
ndwi_2024 = img_2024.normalizedDifference(['B3', 'B8']).rename('NDWI')

ndbi_2019 = img_2019.normalizedDifference(['B11', 'B8']).rename('NDBI')
ndbi_2024 = img_2024.normalizedDifference(['B11', 'B8']).rename('NDBI')

# Summary stats from GEE
for label, img in [('NDVI 2019', ndvi_2019), ('NDVI 2024', ndvi_2024),
                   ('NDBI 2019', ndbi_2019), ('NDBI 2024', ndbi_2024)]:
    val = img.reduceRegion(ee.Reducer.mean(), ROI, 100).getInfo()
    k   = list(val.keys())[0]
    print(f'  {label}: {val[k]:.4f}')

## 4. Visualise with Leafmap (Interactive GEE Tiles)

In [ ]:
m2 = leafmap.Map(center=[12.97, 77.59], zoom=11)
m2.add_basemap('CartoDB.DarkMatter')

ndvi_vis = {'min': -0.2, 'max': 0.8, 'palette': ['red', 'white', 'green']}
ndbi_vis = {'min': -0.3, 'max': 0.5, 'palette': ['white', 'orange', 'red']}

m2.addLayer(ndvi_2019, ndvi_vis, 'NDVI 2019')
m2.addLayer(ndvi_2024, ndvi_vis, 'NDVI 2024')
m2.addLayer(ndbi_2024, ndbi_vis, 'NDBI 2024 (Urban)')
m2

## 5. Change Detection — 2019 → 2024

In [ ]:
delta_ndvi = ndvi_2024.subtract(ndvi_2019).rename('dNDVI')
delta_ndbi = ndbi_2024.subtract(ndbi_2019).rename('dNDBI')

SCALE = 100  # 100m for notebook speed

def gee_mean(img, band):
    return img.reduceRegion(ee.Reducer.mean(), ROI, SCALE).getInfo().get(band)

d_ndvi = gee_mean(delta_ndvi, 'dNDVI')
d_ndbi = gee_mean(delta_ndbi, 'dNDBI')

print(f'ΔNDVI 2019→2024: {d_ndvi:+.4f}  {"⚠️ Vegetation loss" if d_ndvi < 0 else "✅ Vegetation gain"}')
print(f'ΔNDBI 2019→2024: {d_ndbi:+.4f}  {"🏙️  Urban expansion" if d_ndbi > 0 else "stable"}')

# Change map visualization
change_vis = {'min': -0.3, 'max': 0.3, 'palette': ['#d73027', 'white', '#1a9850']}
m3 = leafmap.Map(center=[12.97, 77.59], zoom=11)
m3.add_basemap('CartoDB.DarkMatter')
m3.addLayer(delta_ndvi, change_vis, 'ΔNDVI Change Map')
m3

## 6. SAM 2 Segmentation
> Run `download_tile.py` first to get the satellite patch, then `sam2_segmentation.py` for the full segmentation.
>
> Below we display the pre-generated output.

In [ ]:
from PIL import Image

try:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor='#0f1117')
    for ax in axes: ax.axis('off')
    axes[0].imshow(Image.open('results/bangalore_change_detection.png'))
    axes[0].set_title('Change Detection — 2019 vs 2024', color='white', fontsize=12)
    axes[1].imshow(Image.open('results/bangalore_class_overlay.png'))
    axes[1].set_title('SAM 2 Land Cover Segmentation', color='white', fontsize=12)
    plt.tight_layout()
    plt.savefig('results/notebook_summary.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.show()
except FileNotFoundError:
    print('Run download_tile.py and sam2_segmentation.py first to generate these outputs.')

## 7. Final Summary — Area Statistics

In [ ]:
import json, os
stats_path = 'results/bangalore_class_stats.json'
if os.path.exists(stats_path):
    with open(stats_path) as f:
        stats = json.load(f)
    print('Land Cover Area Statistics (SAM 2 Segmentation):')
    print(f'{"Class":<15} {"Area km²":>10} {"Coverage %":>12}')
    print('-' * 40)
    for cls, s in stats.items():
        print(f'{cls:<15} {s["area_sq_km"]:>10.2f} {s["percentage"]:>11.1f}%')
else:
    print('Stats not yet generated. Run sam2_segmentation.py first.')